# League of Legends Early-Game Advantage and Win Prediction

**Name(s)**: Isabel Yang

**Website Link**: https://iskabell.github.io/LoL-EarlyGame-Win-Prediction/

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

import plotly.express as px
pd.options.plotting.backend = 'plotly'

from dsc80_utils import * # Feel free to uncomment and use this.

## Step 1: Introduction

This project uses professional League of Legends match data from Oracle’s Elixir. 

When first exploring the dataset, I considered several questions:
1. Does being ahead at 15 minutes increase a team’s chance of winning?
2. Which early-game features are most associated with match outcome?
3. Are early objective advantages, like first blood or first tower, strong predictors of winning?

The question I chose to investigate is:

**Does being ahead in the early game increase a team’s probability of winning?**

This question is interesting because early-game advantages such as gold lead, XP lead, and first objectives are important in competitive League of Legends, but I want to examine how strongly those factors are actually associated with winning in the data.

Some of the most relevant columns for this question are:
- `result`: whether the team won or lost the game
- `golddiffat15`: gold difference at 15 minutes
- `xpdiffat15`: experience difference at 15 minutes
- `killsat15`: team kills at 15 minutes
- `opp_killsat15`: opponent kills at 15 minutes
- `firstblood`: whether the team got first blood
- `firsttower`: whether the team got first tower
- `side`: whether the team played on red or blue side

## Step 2: Data Cleaning and Exploratory Data Analysis

To prepare the data for analysis, I first filtered the original dataset to rows where `position == "team"` so that each row corresponds to a team-game observation rather than an individual player. I then engineered `kill_diff_15`, the difference between a team’s kills and its opponent’s kills at 15 minutes, to better capture early-game combat advantage. For modeling and EDA, I selected the columns most relevant to early-game performance and dropped rows with missing values in those columns so that model training and comparisons would be based on complete observations.

In [37]:
# Load dataset and restrict to team-level rows
df_original = pd.read_csv("2025_LoL_esports_match_data_from_OraclesElixir.csv", low_memory=False)
df = df_original[df_original["position"] == "team"].copy()

# Create an early-game kill differential feature and keep columns relevant to the project.
df["kill_diff_15"] = df["killsat15"] - df["opp_killsat15"]

model_df = df[
    [
        "result",
        "side",
        "golddiffat15",
        "xpdiffat15",
        "csdiffat15",
        "kill_diff_15",
        "firstblood",
        "firstdragon",
        "firsttower"
    ]
].dropna()

model_df.head()

,result,side,golddiffat15,xpdiffat15,...,kill_diff_15,firstblood,firstdragon,firsttower
10,0,Blue,-3837.0,-469.0,...,-3.0,0.0,0.0,0.0
11,1,Red,3837.0,469.0,...,3.0,1.0,1.0,1.0
22,1,Blue,5069.0,2014.0,...,5.0,1.0,0.0,1.0
23,0,Red,-5069.0,-2014.0,...,-5.0,0.0,1.0,0.0
34,0,Blue,118.0,1990.0,...,4.0,0.0,0.0,0.0


### Univariate Analysis

In [ ]:
# Histogram showing the distribution of gold difference at 15 minutes
fig = px.histogram(
    model_df,
    x="golddiffat15",
    nbins=50,
    title="Distribution of Gold Difference at 15 Minutes",
    color_discrete_sequence=["#C79B3B"]
)

fig.update_layout(xaxis_title="Gold Difference at 15",
                  yaxis_title="Count"
                 )

fig.show()

In [ ]:
# Histogram showing the distribution of XP difference at 15 minutes
fig = px.histogram(
    model_df,
    x="xpdiffat15",
    nbins=50,
    title="Distribution of XP Difference at 15 Minutes",
    color_discrete_sequence=["#5DA5DA"]
)

fig.update_layout(
    xaxis_title="XP Difference at 15",
    yaxis_title="Count"
)

fig.show()

The distributions of gold difference and XP difference at 15 minutes are both centered near zero, which makes sense because one team’s lead is the other team’s deficit. Both distributions also show a wide spread, suggesting that early-game advantages vary substantially across matches.

### Bivariate Analysis

In [ ]:
# Boxplot of gold difference at 15 minutes by match outcome
fig = px.box(
    model_df,
    x="result",
    y="golddiffat15",
    color="result",
    color_discrete_map={
        0: "#EF553B",
        1: "#00CC96"
    },
    title="Gold Difference at 15 by Match Outcome"
)

fig.update_layout(
    template="plotly_white",
    xaxis_title="Match Result (0 = Loss, 1 = Win)",
    yaxis_title="Gold Difference at 15",
    showlegend=False
)

fig.show()

In [ ]:
# Scatterplot of XP difference vs gold difference at 15 minutes
fig = px.scatter(
    model_df.assign(result_str=model_df["result"].map({0: "Loss", 1: "Win"})),
    x="golddiffat15",
    y="xpdiffat15",
    color="result_str",
    opacity=0.5,
    color_discrete_map={
        "Loss": "#EF553B",
        "Win": "#00CC96"
    },
    title="XP Difference vs Gold Difference at 15"
)

fig.update_traces(marker=dict(size=6))

fig.update_layout(
    template="plotly_white",
    xaxis_title="Gold Diff at 15",
    yaxis_title="XP Diff at 15",
    legend_title_text=""
)

fig.show()

In [ ]:
# overlaid histogram of gold difference at 15 minutes by match outcome
fig = px.histogram(
    model_df,
    x="golddiffat15",
    color="result",
    nbins=50,
    barmode="overlay",
    opacity=0.5,
    title="Gold Difference at 15 by Match Outcome",
    color_discrete_map={
        0: "#EF553B",
        1: "#00CC96"
    }
)

fig.update_layout(xaxis_title="Gold Difference at 15",
                  yaxis_title="Count",
                  )

fig.show()

The bivariate plots show a strong relationship between early-game advantage and match outcome. Winning teams tend to have higher gold differences at 15 minutes, and teams with higher gold and XP differences cluster more heavily among wins than losses.

### Interesting Aggregates

In [ ]:
# Bar chart of win rate by first tower secured 
win_rates = model_df.groupby("firsttower")["result"].mean().reset_index()
win_rates["firsttower"] = win_rates["firsttower"].map({
    0.0: "No",
    1.0: "Yes"
})

fig = px.bar(
    win_rates,
    x="firsttower",
    y="result",
    color="firsttower",
    color_discrete_map={
        "No": "#EF553B",
        "Yes": "#00CC96"
    },
    title="Win Rate by First Tower Secured"
)

fig.update_traces(texttemplate="%{y:.2f}", textposition="outside")

fig.update_layout(
    template="plotly_white",
    xaxis_title="First Tower",
    yaxis_title="Win Rate",
    showlegend=False
)

fig.show()

In [ ]:
# Pivot table showing win rate by first tower and gold difference bucket
pivot_table = model_df.pivot_table(
    values="result",
    index="firsttower",
    columns="gold_bucket",
    aggfunc="mean",
    observed=False
)

pivot_table

gold_bucket,Large Deficit,Small Deficit,Even,Small Lead,Large Lead
firsttower,,,,,
0.0,0.11,0.28,0.41,0.57,0.78
1.0,0.22,0.43,0.59,0.72,0.89


In [ ]:
# Bin teams by gold difference at 15 minutes and compute the win rate within each bucket
model_df["gold_bucket"] = pd.cut(
    model_df["golddiffat15"],
    bins=[-10000, -2000, -500, 500, 2000, 10000],
    labels=["Large Deficit", "Small Deficit", "Even", "Small Lead", "Large Lead"]
)

bucket_win_rate = (
    model_df.groupby("gold_bucket", observed=False)["result"]
    .mean()
    .reset_index()
)

bucket_win_rate

,gold_bucket,result
0,Large Deficit,0.12
1,Small Deficit,0.33
2,Even,0.50
3,Small Lead,0.67
4,Large Lead,0.88


In [ ]:
# Bar chart of win rates across gold difference buckets
fig = px.bar(
    bucket_win_rate,
    x="gold_bucket",
    y="result",
    title="Win Rate by Gold Difference at 15 Buckets"
)

fig.update_traces(texttemplate="%{y:.0%}", textposition="outside")
fig.update_layout(template="plotly_white", yaxis_range=[0,1])

fig.show()

The aggregate summaries show a strong and consistent relationship between early-game advantage and winning. Teams with larger gold leads at 15 minutes have much higher win rates, while teams with gold deficits win much less often, suggesting that early economic control is closely tied to match outcome. The grouped results also show that securing first tower is associated with a higher win rate across multiple gold-difference buckets, which suggests that early objective control may be associated with match outcome as well. 

## Step 3: Assessment of Missingness

### Missingness Dependency

I analyzed the missingness of `deathsat25`, which is the number of deaths a team has at 25 minutes. This column has non-trivial missingness because some games end before the 25-minute mark, so 25-minute statistics are not always observed.

To study this missingness, I created a Boolean column `deathsat25_missing` and used permutation tests to determine whether its missingness depends on other variables. I tested whether the missingness depends on `short_game`, which indicates whether a game lasted fewer than 25 minutes, and on `result`, which indicates whether the team won or lost.

In [64]:
# Restrict to team-level rows and create indicators used in missingness analysis
df_team = df_original[df_original["position"] == "team"].copy()
df_team["gamelength_min"] = df_team["gamelength"] / 60
df_team["short_game"] = df_team["gamelength_min"] < 25
df_team["deathsat25_missing"] = df_team["deathsat25"].isna()

print("Counts of short vs non-short games:")
print(df_team["short_game"].value_counts())
print("\nMissingness rate of deathsat25 by short_game:")
print(df_team.groupby("short_game")["deathsat25_missing"].mean())

Counts of short vs non-short games:
short_game
False    18990
True      1116
Name: count, dtype: int64

Missingness rate of deathsat25 by short_game:
short_game
False    0.08
True     0.66
Name: deathsat25_missing, dtype: float64


For the first permutation test:

- **Null hypothesis:** The missingness of `deathsat25` does not depend on whether a game is short.
- **Alternative hypothesis:** The missingness of `deathsat25` does depend on whether a game is short.

I use the difference in missingness rates between short and non-short games as my test statistic.

In [65]:
observed_diff = (
    df_team.groupby("short_game")["deathsat25_missing"].mean().iloc[1]
    - df_team.groupby("short_game")["deathsat25_missing"].mean().iloc[0]
)

perms = []
for _ in range(1000):
    shuffled = np.random.permutation(df_team["short_game"])
    
    perm_diff = (
        df_team.assign(shuffled_short=shuffled)
        .groupby("shuffled_short")["deathsat25_missing"]
        .mean()
        .iloc[1]
        - df_team.assign(shuffled_short=shuffled)
        .groupby("shuffled_short")["deathsat25_missing"]
        .mean()
        .iloc[0]
    )
    
    perms.append(perm_diff)

p_value = np.mean(np.abs(perms) >= abs(observed_diff))
p_value

0.0

The p-value is 0.0, so I reject the null hypothesis. There is strong statistical evidence that the missingness of `deathsat25` depends on whether the game is short.

For the second permutation test:

- **Null hypothesis:** The missingness of `deathsat25` does not depend on match result.
- **Alternative hypothesis:** The missingness of `deathsat25` does depend on match result.

I use the difference in missingness rates between the two groups as my test statistic.

In [66]:
observed_diff = (
    df_team.groupby("result")["deathsat25_missing"].mean().iloc[1]
    - df_team.groupby("result")["deathsat25_missing"].mean().iloc[0]
)

perms = []

for _ in range(1000):
    shuffled = np.random.permutation(df_team["result"])
    
    shuffled_means = (
        df_team.assign(shuffled_result=shuffled)
        .groupby("shuffled_result")["deathsat25_missing"]
        .mean()
    )
    
    perm_diff = shuffled_means.iloc[1] - shuffled_means.iloc[0]
    
    perms.append(perm_diff)

p_value = np.mean(np.abs(perms) >= abs(observed_diff))

p_value

1.0

The p-value is 1.0, so I fail to reject the null hypothesis. There is not enough statistical evidence that the missingness of `deathsat25` does not depend on match result.

## Step 4: Hypothesis Testing

- **Null hypothesis:** Teams that are ahead in gold at 15 minutes and teams that are not ahead at 15 minutes have the same win rate.
- **Alternative hypothesis:** Teams that are ahead in gold at 15 minutes have a higher win rate than teams that are not ahead at 15 minutes.

I use the difference in win rates between the two groups as my test statistic, and I use a significance level of 0.05. This is a good test statistic because my question is specifically about whether early gold advantage is associated with a higher probability of winning.

In [ ]:
# Compare observed win rates for teams ahead vs not ahead in gold at 15 minutes
df_team["ahead15"] = df_team["golddiffat15"] > 0

observed = df_team.groupby("ahead15")["result"].mean()
observed_diff = observed.iloc[1] - observed.iloc[0]

print("Observed win rates by ahead15:")
print(observed)
print("\nObserved difference in win rate:", observed_diff)

Observed win rates by ahead15:
ahead15
False    0.31
True     0.73
Name: result, dtype: float64

Observed difference in win rate: 0.42327394397366724


In [73]:
# One-sided permutation test for the difference in win rates
perms = []

for _ in range(1000):
    shuffled = np.random.permutation(df_team["result"])
    
    shuffled_means = (
        df_team.assign(shuffled_result=shuffled)
        .groupby("ahead15")["shuffled_result"]
        .mean()
    )
    
    perm_diff = shuffled_means.iloc[1] - shuffled_means.iloc[0]
    perms.append(perm_diff)

p_value = np.mean(np.array(perms) >= observed_diff)
p_value

0.0

The observed win rate for teams that were not ahead in gold at 15 minutes was 0.31, while the observed win rate for teams that were ahead in gold at 15 minutes was 0.73. The observed difference in win rates was about 0.423. The permutation test resulted in a p-value of 0.0, so I reject the null hypothesis. There is strong statistical evidence that teams ahead in gold at 15 minutes win more often than teams that are not ahead at 15 minutes.

## Step 5: Framing a Prediction Problem

My prediction problem is **binary classification**: predicting whether a team will win a match (`result = 1`) or lose a match (`result = 0`) using only information available by 15 minutes into the game.

I chose `result` as the response variable because match outcome is the most natural quantity to predict in this dataset, and my overall project focuses on whether early-game advantages are associated with winning. 

I evaluate my models using **accuracy**. Since this is a binary classification problem and the classes are reasonably balanced at the team level, accuracy is an appropriate metric for measuring overall predictive performance on unseen data. I chose accuracy over metrics like precision or recall because, for this task, I am most interested in how often the model correctly predicts the overall game outcome rather than optimizing performance for one class specifically.

At the **time of prediction**, I assume I would only know features available by 15 minutes into the game. For that reason, I only use early-game variables such as `golddiffat15`, `xpdiffat15`, `csdiffat15`, `firstblood`, `firstdragon`, `firsttower`, and engineered features derived from these early-game statistics. I do not use any information from later in the game, since it would not be available when making the prediction.

## Step 6: Baseline Model

For my baseline model, I use a **logistic regression classifier** to predict whether a team wins the match (`result = 1`) or loses (`result = 0`) based on early-game information available by 15 minutes.

The model uses four features:
- `golddiffat15` (**quantitative**): gold difference at 15 minutes
- `xpdiffat15` (**quantitative**): experience difference at 15 minutes
- `firstblood` (**binary / nominal indicator**): whether the team secured first blood
- `firsttower` (**binary / nominal indicator**): whether the team secured first tower

I standardize the quantitative features using `StandardScaler` and pass the binary indicator features through unchanged. All preprocessing and model fitting are implemented in a single sklearn `Pipeline`.

I evaluate the baseline model using **accuracy** on an unseen test set to measure how well it generalizes to unseen games.

In [31]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

features = [
    "golddiffat15",
    "xpdiffat15",
    "firstblood",
    "firsttower"
]

X = model_df[features]
y = model_df["result"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric = ["golddiffat15", "xpdiffat15"]
binary = ["firstblood", "firsttower"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("bin", "passthrough", binary)
])

baseline_model = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression())
])

baseline_model.fit(X_train, y_train)

preds = baseline_model.predict(X_test)
accuracy = accuracy_score(y_test, preds)

print("Baseline Accuracy:", accuracy)

Baseline Accuracy: 0.7415426251691475


The baseline model achieved an accuracy of about **0.7415** on the unseen test set. This suggests that early-game features such as gold difference, XP difference, and first objectives provides substantial information about match outcome. The final model will improveme the baseline model through feature engineering and hyperparameter tuning.

## Step 7: Final Model

For my final model, I continued using **logistic regression** but added two engineered features to better capture early-game advantage:

- `kill_diff_15`: the difference between a team’s kills and its opponent’s kills at 15 minutes
- `gold_xp_interaction`: the product of `golddiffat15` and `xpdiffat15`

I expected `kill_diff_15` to help because early kill advantage reflects combat control that may not be fully captured by gold and XP alone. I expected `gold_xp_interaction` to help because teams that are ahead in both gold and experience at the same time may be in a stronger overall position than teams with a lead in only one of these measures.

Before tuning, I planned to search over the hyperparameters `C` and `class_weight` in logistic regression.

- `C` controls the strength of regularization. Smaller values apply stronger regularization, while larger values allow the model to fit the training data more closely.
- `class_weight` changes how the model penalizes misclassification across classes, which may help if one class is slightly harder to predict or if the model benefits from reweighting wins and losses.

I chose to tune these hyperparameters because my final model includes additional engineered features, so selecting an appropriate regularization level and class weighting may improve generalization to unseen data.

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Engineer new early-game features, tune the logistic regression model, and evaluate it on the unseen test set.

df["kill_diff_15"] = df["killsat15"] - df["opp_killsat15"]
df["gold_xp_interaction"] = df["golddiffat15"] * df["xpdiffat15"]

model_df = df[
    [
        "result",
        "golddiffat15",
        "xpdiffat15",
        "firstblood",
        "firsttower",
        "kill_diff_15",
        "gold_xp_interaction"
    ]
].dropna()

features = [
    "golddiffat15",
    "xpdiffat15",
    "firstblood",
    "firsttower",
    "kill_diff_15",
    "gold_xp_interaction"
]

X = model_df[features]
y = model_df["result"]

# same split as baseline
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

numeric = [
    "golddiffat15",
    "xpdiffat15",
    "kill_diff_15",
    "gold_xp_interaction"
]
binary = ["firstblood", "firsttower"]

preprocess = ColumnTransformer([
    ("num", StandardScaler(), numeric),
    ("bin", "passthrough", binary)
])

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=2000))
])

param_grid = {
    "model__C": [0.01, 0.1, 1, 10],
    "model__class_weight": [None, "balanced"]
}

grid = GridSearchCV(pipe, param_grid, cv=5, scoring="accuracy")
grid.fit(X_train, y_train)

best_model = grid.best_estimator_
preds = best_model.predict(X_test)
accuracy = accuracy_score(y_test, preds)

print("Best Params:", grid.best_params_)
print("Final Model Accuracy:", accuracy)

Best Params: {'model__C': 0.1, 'model__class_weight': 'balanced'}
Final Model Accuracy: 0.7423545331529093


The final model achieved an accuracy of about **0.74235**, which is a small improvement over the baseline accuracy of about **0.74154**. 

The best-performing hyperparameters were `C = 0.1` and `class_weight = "balanced"`. This suggests that a moderate amount of regularization and reweighting of the classes helped the model generalize slightly better to unseen data.

## Step 8: Fairness Analysis

To assess fairness, I test whether my final model performs worse for red-side teams than for blue-side teams.

- **Group X:** red-side teams  
- **Group Y:** blue-side teams  
- **Evaluation metric:** precision  
- **Null hypothesis:** My model is fair with respect to side. Its precision for red-side teams and blue-side teams is roughly the same.
- **Alternative hypothesis:** My model is unfair with respect to side. Its precision for red-side teams is lower than its precision for blue-side teams.  
- **Test statistic:** precision(red) − precision(blue)  
- **Significance level:** 0.05  

I chose side as the grouping variable because red side and blue side are meaningful groups in League of Legends, and I want to check whether the model performs differently for one side than the other.

In [ ]:
# Evaluate whether the final fitted model has different precision on red-side and blue-side teams
from sklearn.metrics import precision_score
import numpy as np

# Get side labels for the same unseen test rows used in Step 7
test_results = df.loc[X_test.index, ["side"]].copy()
test_results["y_true"] = y_test
test_results["y_pred"] = best_model.predict(X_test)

red = test_results[test_results["side"] == "Red"]
blue = test_results[test_results["side"] == "Blue"]

red_precision = precision_score(red["y_true"], red["y_pred"], zero_division=0)
blue_precision = precision_score(blue["y_true"], blue["y_pred"], zero_division=0)

observed_diff = red_precision - blue_precision

print("Red precision:", red_precision)
print("Blue precision:", blue_precision)
print("Observed difference (Red - Blue):", observed_diff)

# One-sided permutation test
n_simulations = 5000
diffs = []

for _ in range(n_simulations):
    shuffled_groups = test_results["side"].sample(frac=1, replace=False).values

    shuffled_df = test_results.copy()
    shuffled_df["shuffled_side"] = shuffled_groups

    red_shuffled = shuffled_df[shuffled_df["shuffled_side"] == "Red"]
    blue_shuffled = shuffled_df[shuffled_df["shuffled_side"] == "Blue"]

    red_prec = precision_score(red_shuffled["y_true"], red_shuffled["y_pred"], zero_division=0)
    blue_prec = precision_score(blue_shuffled["y_true"], blue_shuffled["y_pred"], zero_division=0)

    diffs.append(red_prec - blue_prec)

diffs = np.array(diffs)
p_value = np.mean(diffs <= observed_diff)

print("p-value:", p_value)

Red precision: 0.7400228050171037
Blue precision: 0.7585139318885449
Observed difference (Red - Blue): -0.018491126871441166
p-value: 0.1728


The final model had a precision of **0.7400** for red-side teams and **0.7585** for blue-side teams, so the observed difference in precision was **-0.0185**. The permutation test gave a p-value of **0.1728**, which is greater than **0.05**. As a result, I fail to reject the null hypothesis and do not have sufficient evidence that the model is less precise for red-side teams than for blue-side teams.